# Comparing ensemble models for tourist arrivals

I tried more models with the same data and time splits. This extra experiment does not change the deployed model.

Settings are chosen on 2023 after training through 2022. Models then train through 2023 for the 2024 comparison. The later 2025-September 2026 period has already been examined, so it is not an untouched test set.

MAE is the average error in arrivals. R2 measures explained variation, not percentage accuracy. WAPE is total absolute error divided by total actual arrivals.

In [ ]:
%pip install -q scikit-learn==1.9.1 pandas==3.0.6 numpy==2.5.4 catboost==1.2.10 xgboost==3.4.1 lightgbm==4.7.0 interpret-core==0.7.8

## Models and checks

I compare bagged trees, Random Forest, Extra Trees, seven boosting approaches, two voting combinations and stacking. Ridge is only the stacking combiner. Most models try two settings; the combined models use fixed base settings. This is a small search.

Stacking learns from earlier-year predictions. The recursive check starts in January, April, July and October 2024. Later steps reuse predictions where actual lags would not yet be known. CatBoost and EBM handle categories directly; other models use the existing one-hot preprocessing.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
import sys
import json
import time
import hashlib
import warnings
import importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor, BaggingRegressor,
                              HistGradientBoostingRegressor, GradientBoostingRegressor,
                              AdaBoostRegressor, VotingRegressor)
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from interpret.glassbox import ExplainableBoostingRegressor
warnings.filterwarnings('ignore', message='X does not have valid feature names')

dataset_path = Path('../data/Tourism_Arrivals.csv')
if not dataset_path.exists():
    dataset_path = Path('Tourism_Arrivals.csv')
if not dataset_path.exists():
    from google.colab import files
    files.upload()

output = Path('.')
features = ['Country', 'lag_1', 'lag_2', 'lag_3', 'lag_12', 'month_number']
data = pd.read_csv(dataset_path)
data['Month'] = pd.PeriodIndex(data.Year_Month, freq='M')
run_rows, failures = [], []
selected = {}
fitted_cache = {}


def scores(actual, prediction):
    actual, prediction = np.asarray(actual), np.asarray(prediction)
    assert np.isfinite(prediction).all()
    return {'MAE': mean_absolute_error(actual, prediction),
            'RMSE': root_mean_squared_error(actual, prediction),
            'R2': r2_score(actual, prediction),
            'WAPE_pct': 100 * np.abs(actual - prediction).sum() / actual.sum()}


def pipeline(model):
    preprocessing = ColumnTransformer([
        ('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Country', 'month_number']),
        ('history', StandardScaler(), ['lag_1', 'lag_2', 'lag_3', 'lag_12'])])
    return make_pipeline(preprocessing, model)


def cat_frame(frame):
    result = frame.copy()
    result['Country'] = result.Country.astype(str)
    result['month_number'] = result.month_number.astype(str)
    return result


options = {
    'Extra Trees': [{'max_depth': 8, 'min_samples_leaf': 10}, {'max_depth': 12, 'min_samples_leaf': 5}],
    'Random Forest': [{'max_depth': 12, 'min_samples_leaf': 5}, {'max_depth': 20, 'min_samples_leaf': 2}],
    'Bagged trees': [{'max_depth': 8, 'min_samples_leaf': 10}, {'max_depth': 12, 'min_samples_leaf': 5}],
    'Histogram boosting': [{'max_iter': 150, 'max_leaf_nodes': 15}, {'max_iter': 250, 'max_leaf_nodes': 31}],
    'Gradient boosting': [{'max_depth': 2}, {'max_depth': 3}],
    'AdaBoost': [{'learning_rate': .05}, {'learning_rate': .2}],
    'XGBoost': [{'max_depth': 3}, {'max_depth': 5}],
    'LightGBM': [{'num_leaves': 15}, {'num_leaves': 31}],
    'CatBoost': [{'depth': 4}, {'depth': 6}],
    'EBM': [{'interactions': 0}, {'interactions': 5}],
    'Voting (forest + histogram)': [{}],
    'Voting (forest + CatBoost + LightGBM)': [{}],
    'Time-aware stacking': [{'alpha': 10}, {'alpha': 10000}]
}
families = {'Extra Trees': 'Randomized trees', 'Random Forest': 'Bagging', 'Bagged trees': 'Bagging',
            'Voting (forest + histogram)': 'Voting', 'Voting (forest + CatBoost + LightGBM)': 'Voting',
            'Time-aware stacking': 'Stacking'}


def make_model(name, config):
    if name == 'Extra Trees':
        return pipeline(ExtraTreesRegressor(n_estimators=150, max_features=1.0, bootstrap=False,
                                           random_state=42, n_jobs=4, **config))
    if name == 'Random Forest':
        return pipeline(RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=4, **config))
    if name == 'Bagged trees':
        tree = DecisionTreeRegressor(random_state=42, **config)
        return pipeline(BaggingRegressor(estimator=tree, n_estimators=150, max_samples=.8,
                                         bootstrap=True, random_state=42, n_jobs=2))
    if name == 'Histogram boosting':
        return pipeline(HistGradientBoostingRegressor(l2_regularization=10, random_state=42, **config))
    if name == 'Gradient boosting':
        return pipeline(GradientBoostingRegressor(n_estimators=300, learning_rate=.05,
                                                  min_samples_leaf=10, random_state=42, **config))
    if name == 'AdaBoost':
        return pipeline(AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=3, min_samples_leaf=10),
                                         n_estimators=150, loss='linear', random_state=42, **config))
    if name == 'XGBoost':
        return pipeline(XGBRegressor(n_estimators=400, learning_rate=.05, min_child_weight=10,
                                     reg_lambda=10, subsample=.8, colsample_bytree=.8,
                                     objective='reg:squarederror', tree_method='hist',
                                     random_state=42, n_jobs=4, **config))
    if name == 'LightGBM':
        return pipeline(LGBMRegressor(n_estimators=400, learning_rate=.05, min_child_samples=30,
                                      reg_lambda=10, random_state=42, n_jobs=4, verbosity=-1,
                                      deterministic=True, force_col_wise=True, **config))
    if name == 'CatBoost':
        return make_pipeline(FunctionTransformer(cat_frame), CatBoostRegressor(
            iterations=400, learning_rate=.05, l2_leaf_reg=5, loss_function='RMSE',
            cat_features=['Country', 'month_number'], random_seed=42, thread_count=4,
            verbose=False, allow_writing_files=False, **config))
    if name == 'EBM':
        return ExplainableBoostingRegressor(
            feature_types=['nominal', 'continuous', 'continuous', 'continuous', 'continuous', 'nominal'],
            max_rounds=1500, outer_bags=1, validation_size=0, early_stopping_rounds=0,
            max_bins=128, learning_rate=.03, min_samples_leaf=10,
            random_state=42, n_jobs=2, **config)
    if name.startswith('Voting'):
        bases = (['Random Forest', 'Histogram boosting'] if name.endswith('histogram)')
                 else ['Random Forest', 'CatBoost', 'LightGBM'])
        return VotingRegressor([(str(i), make_model(base, options[base][0])) for i, base in enumerate(bases)])
    raise ValueError(name)


def base_fit(name, cutoff):
    key = (name, cutoff)
    if key not in fitted_cache:
        earlier = data.loc[data.Month.dt.year.le(cutoff)]
        fitted_cache[key] = make_model(name, options[name][0]).fit(earlier[features], earlier.Arrivals)
    return fitted_cache[key]


def fit_stacker(train, alpha):
    # Each complete year is predicted using only earlier years.
    bases = ['Random Forest', 'CatBoost', 'LightGBM']
    oof, actual = [], []
    cutoff = int(train.Month.dt.year.max())
    for year in range(2021, cutoff + 1):
        check = train.loc[train.Month.dt.year.eq(year)]
        models = [base_fit(name, year - 1) for name in bases]
        oof.append(np.column_stack([np.maximum(0, model.predict(check[features])) for model in models]))
        actual.append(check.Arrivals.to_numpy())
    combiner = Ridge(alpha=alpha).fit(np.vstack(oof), np.concatenate(actual))
    full_models = [base_fit(name, cutoff) for name in bases]
    return {'bases': full_models, 'combiner': combiner}


def fit_model(name, config, train):
    if name == 'Time-aware stacking':
        return fit_stacker(train, config['alpha'])
    if name == 'Voting (forest + CatBoost + LightGBM)':
        bases = [make_model(base, options[base][0]).fit(train[features], train.Arrivals)
                 for base in ['Random Forest', 'CatBoost', 'LightGBM']]
        return {'bases': bases, 'combiner': None}
    return make_model(name, config).fit(train[features], train.Arrivals)


def predict(model, frame):
    if isinstance(model, dict):
        estimates = np.column_stack([np.maximum(0, base.predict(frame[features])) for base in model['bases']])
        values = estimates.mean(axis=1) if model['combiner'] is None else model['combiner'].predict(estimates)
    else:
        values = model.predict(frame[features])
    return np.maximum(0, values)


def save_row(name, stage, actual, prediction, config, horizon=1):
    row = {'Model': name, 'Family': families.get(name, 'Boosting'), 'Stage': stage,
           'Horizon': horizon, 'Cases': len(actual), **scores(actual, prediction),
           'Settings': json.dumps(config, sort_keys=True)}
    run_rows.append(row)
    pd.DataFrame(run_rows).to_csv(output / 'Model_Comparison_Results.csv', index=False)
    print(f'{name} | {stage} | h={horizon} | MAE={row["MAE"]:.3f} | R2={row["R2"]:.4f}', flush=True)
    return row


def recursive_check(model, name):
    history = data.set_index(['Country', 'Month']).Arrivals
    by_horizon = {h: [[], []] for h in [1, 2, 3]}
    countries = sorted(data.loc[data.Month.dt.year.le(2023), 'Country'].unique())
    for start in [pd.Period(year=2024, month=m, freq='M') for m in [1, 4, 7, 10]]:
        earlier_predictions = {}
        for step in range(3):
            target = start + step
            batch = []
            for country in countries:
                row = {'Country': country, 'month_number': target.month}
                for lag in [1, 2, 3, 12]:
                    key = (country, target - lag)
                    row[f'lag_{lag}'] = (earlier_predictions.get(key) if target - lag >= start else history.get(key))
                if all(pd.notna(value) for value in row.values()):
                    batch.append(row)
            frame = pd.DataFrame(batch)
            if name == 'Previous month':
                estimates = frame.lag_1.to_numpy()
            elif name == 'Same month last year':
                estimates = frame.lag_12.to_numpy()
            else:
                estimates = predict(model, frame)
            for country, estimate in zip(frame.Country, estimates):
                earlier_predictions[(country, target)] = estimate
                actual = history.get((country, target))
                if pd.notna(actual):
                    by_horizon[step + 1][0].append(actual)
                    by_horizon[step + 1][1].append(estimate)
    for horizon, (actual, estimates) in by_horizon.items():
        save_row(name, 'Recursive 2024', actual, estimates, selected.get(name, {}), horizon)


def run_comparison():
    tuning_train = data.loc[data.Month.dt.year.le(2022)]
    tuning_check = data.loc[data.Month.dt.year.eq(2023)]
    train = data.loc[data.Month.dt.year.le(2023)]
    validation = data.loc[data.Month.dt.year.eq(2024) & data.Country.isin(train.Country.unique())]
    later_train = data.loc[data.Month.dt.year.le(2024)]
    later = data.loc[data.Month.dt.year.ge(2025) & data.Country.isin(later_train.Country.unique())]
    assert tuning_train.Month.max() < tuning_check.Month.min() < validation.Month.min() < later.Month.min()
    for name, configs in options.items():
        try:
            choices = []
            for config in configs:
                model = fit_model(name, config, tuning_train)
                choices.append(save_row(name, 'Tuning 2023', tuning_check.Arrivals, predict(model, tuning_check), config)['MAE'])
            selected[name] = configs[int(np.argmin(choices))]
            model = fit_model(name, selected[name], train)
            save_row(name, 'Validation 2024', validation.Arrivals, predict(model, validation), selected[name])
            recursive_check(model, name)
            later_model = fit_model(name, selected[name], later_train)
            save_row(name, 'Later 2025-2026 (previously examined)', later.Arrivals, predict(later_model, later), selected[name])
        except Exception as error:
            failures.append({'model': name, 'error': str(error), 'type': type(error).__name__})
            print(f'{name} FAILED: {type(error).__name__}: {error}', flush=True)
    for name, column in [('Previous month', 'lag_1'), ('Same month last year', 'lag_12')]:
        families[name] = 'Baseline'
        save_row(name, 'Validation 2024', validation.Arrivals, validation[column].to_numpy(), {})
        save_row(name, 'Later 2025-2026 (previously examined)', later.Arrivals, later[column].to_numpy(), {})
        recursive_check(None, name)
    return pd.DataFrame(run_rows)




In [ ]:
results = run_comparison()
results.to_csv("Model_Comparison_Results.csv", index=False)

## Results

These tables come from the completed local run. Running the cells above repeats the comparison.

### Validation 2024

In [ ]:
results.loc[results.Stage.eq('Validation 2024'), ['Model', 'Cases', 'MAE', 'RMSE', 'R2', 'WAPE_pct']].sort_values("MAE").round(3)

Model,Cases,MAE,RMSE,R2,WAPE_pct
Voting (forest + CatBoost + LightGBM),2207,227.099,919.955,0.934,24.409
CatBoost,2207,232.301,873.159,0.941,24.969
Random Forest,2207,235.840,941.069,0.931,25.349
Bagged trees,2207,240.605,1008.358,0.921,25.861
Voting (forest + histogram),2207,243.627,1006.892,0.921,26.186
LightGBM,2207,252.439,1005.557,0.921,27.133
XGBoost,2207,253.725,980.224,0.925,27.271
Gradient boosting,2207,257.200,1002.224,0.922,27.645
Extra Trees,2207,258.393,1111.300,0.904,27.773
Histogram boosting,2207,260.443,1067.728,0.911,27.993


### Later 2025-2026 (previously examined)

In [ ]:
results.loc[results.Stage.eq('Later 2025-2026 (previously examined)'), ['Model', 'Cases', 'MAE', 'RMSE', 'R2', 'WAPE_pct']].sort_values("MAE").round(3)

Model,Cases,MAE,RMSE,R2,WAPE_pct
Same month last year,3978,219.670,972.141,0.947,21.544
Random Forest,3978,238.997,1089.767,0.933,23.439
Voting (forest + CatBoost + LightGBM),3978,246.638,1173.814,0.922,24.189
Voting (forest + histogram),3978,248.394,1170.093,0.923,24.361
Bagged trees,3978,252.486,1163.063,0.924,24.762
LightGBM,3978,265.794,1269.968,0.909,26.067
CatBoost,3978,268.480,1214.140,0.917,26.331
Histogram boosting,3978,269.273,1250.511,0.912,26.409
XGBoost,3978,277.359,1301.093,0.905,27.202
Time-aware stacking,3978,281.892,1054.670,0.937,27.646


### Three-month MAE

Each column is a forecast step. Lower is better.

In [ ]:
results.loc[results.Stage.eq("Recursive 2024")].pivot(index="Model", columns="Horizon", values="MAE").round(2)

Horizon,1,2,3
Model,,,
AdaBoost,381.16,482.98,482.19
Bagged trees,281.96,329.35,359.50
CatBoost,273.38,331.89,341.72
EBM,303.42,378.31,441.81
Extra Trees,286.69,353.78,386.80
Gradient boosting,282.89,346.82,399.24
Histogram boosting,268.59,346.16,362.20
LightGBM,258.97,334.83,366.94
Previous month,351.47,463.25,484.00


## What I found

The forest/CatBoost/LightGBM vote has the lowest 2024 MAE (227.10). CatBoost has the highest R2 (0.941). Tuned Random Forest does better than these models in the later period. Stacking performs worse, especially over three months.

The same-month-last-year baseline is still better in the later period and at the third forecast step. These extra models have not solved that limitation. Repeated comparisons make 2024 development evidence rather than an independent final test.

Before replacing the main model, I need to check the new-algorithm requirement. XGBoost, LightGBM, CatBoost and EBM are mentioned in the course material. Better scores alone do not settle eligibility.

The first three-model voting run hit a CatBoost/sklearn cloning error. I fitted the models separately and averaged predictions instead. EBM uses one outer bag because no internal validation split is used.

Documentation: [CatBoost](https://catboost.ai/docs/en/concepts/python-reference_catboostregressor), [LightGBM](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMRegressor.html), [XGBoost](https://xgboost.readthedocs.io/en/latest/python/python_api.html), [EBM](https://interpret.ml/docs/python/api/ExplainableBoostingRegressor.html).
